# Práctica: Correlación y Regresión Lineal

En este cuaderno calculamos correlaciones, ajustamos un modelo de regresión lineal simple, evaluamos su bondad de ajuste y contrastamos si la relación encontrada es estadísticamente significativa.

In [ ]:
import sys
sys.path.append("../../src")

import numpy as np
import matplotlib.pyplot as plt
from stats_toolkit import regression as reg

## 1. Datos: inversión en publicidad vs. ventas

Simulamos 20 meses de inversión en publicidad (miles de €) y ventas resultantes (miles de €), con una relación lineal real más algo de ruido.

In [ ]:
rng = np.random.default_rng(3)
publicidad = rng.uniform(5, 50, size=20)
ventas = 20 + 3.5 * publicidad + rng.normal(0, 25, size=20)

plt.scatter(publicidad, ventas, color="steelblue")
plt.xlabel("Inversión en publicidad (miles de €)")
plt.ylabel("Ventas (miles de €)")
plt.title("Publicidad vs. Ventas")
plt.show()

## 2. Correlación

In [ ]:
r = reg.correlation(publicidad, ventas)
print(f"Correlación de Pearson: r = {r:.3f}")

## 3. Ajuste del modelo de regresión

In [ ]:
beta0, beta1 = reg.simple_linear_regression(publicidad, ventas)
print(f"Ventas estimadas = {beta0:.2f} + {beta1:.2f} × Publicidad")

x_line = np.linspace(publicidad.min(), publicidad.max(), 100)
y_line = reg.predict(x_line, beta0, beta1)

plt.scatter(publicidad, ventas, color="steelblue", label="Datos observados")
plt.plot(x_line, y_line, color="crimson", label="Recta ajustada")
plt.xlabel("Inversión en publicidad (miles de €)")
plt.ylabel("Ventas (miles de €)")
plt.legend()
plt.title("Regresión lineal: Publicidad vs. Ventas")
plt.show()

## 4. Residuos y bondad de ajuste

Un buen ajuste debería mostrar residuos dispersos aleatoriamente alrededor de cero, sin ningún patrón sistemático (si hubiera un patrón claro, indicaría que el modelo lineal no captura bien la relación).

In [ ]:
resid = reg.residuals(publicidad, ventas, beta0, beta1)

plt.scatter(publicidad, resid, color="darkorange")
plt.axhline(0, color="black", linestyle="--")
plt.xlabel("Inversión en publicidad (miles de €)")
plt.ylabel("Residuo (ventas observadas - predichas)")
plt.title("Gráfico de residuos")
plt.show()

r2 = reg.r_squared(publicidad, ventas, beta0, beta1)
print(f"R² = {r2:.3f} — el modelo explica el {r2*100:.1f}% de la variabilidad de las ventas")

## 5. ¿Es la relación estadísticamente significativa?

Contrastamos H0: β1 = 0 (no hay relación real entre publicidad y ventas) frente a H1: β1 ≠ 0.

In [ ]:
t_stat, p_value = reg.t_test_slope(publicidad, ventas, beta0, beta1)
lower, upper = reg.confidence_interval_slope(publicidad, ventas, beta0, beta1, confidence=0.95)

print(f"t = {t_stat:.3f}, p-valor = {p_value:.6f}")
print(f"IC 95% para β1: ({lower:.3f}, {upper:.3f})")

if p_value < 0.05:
    print("\nConclusión: la relación entre publicidad y ventas es estadísticamente significativa.")
else:
    print("\nConclusión: no hay evidencia suficiente de una relación lineal real.")

## Ejercicios propuestos

1. Genera un segundo conjunto de datos donde `ventas` sea prácticamente independiente de `publicidad` (por ejemplo, ruido puro sin relación real) y repite todo el análisis. Compara el R² y el p-valor con los obtenidos aquí.
2. Usa el modelo ajustado para predecir las ventas esperadas con una inversión en publicidad de 60.000 €. ¿Te fías igual de esta predicción que de una dentro del rango de datos observados (5-50)? ¿Por qué?
3. Calcula la correlación entre publicidad y ventas usando `numpy.corrcoef` y comprueba que coincide con el resultado de `reg.correlation`. Comprueba también que `r ** 2` coincide con el `R²` calculado en la sección 4.